In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv(r'outputs\seller_scoring_bgnbd_gammagamma_evaluation.csv')

In [4]:
df

,seller_id,frequency,recency,T,monetary_value,total_orders_cal,total_gmv_cal,historical_aov_cal,prob_alive,expected_avg_gmv,...,actual_commission_60d,expected_orders_90d,activity_score_90d,p_purchase_approx_90d,predicted_gmv_90d,expected_commission_90d,actual_orders_90d,actual_active_90d,actual_gmv_90d,actual_commission_90d
0,0015a82c2db000af6aaaf3ae2ecb0532,2,21.416308,216.071458,895.000000,3,2685.00,895.000000,9.916166e-02,894.276630,...,0.0000,9.741976e-02,9.741976e-02,9.282487e-02,87.120216,1.306803e+01,0.0,0,0.00,0.0000
1,001cca7ae9ae17fb1caed9dfb1094831,191,447.554861,450.204109,125.739424,192,24116.13,125.604844,9.977010e-01,125.777572,...,36.1500,3.739724e+01,3.739724e+01,1.000000e+00,4703.734589,7.055602e+02,3.0,1,370.90,55.6350
2,002100f778ceb8431b7a1020ff7ab48f,49,210.498519,228.957963,24.626531,50,1216.60,24.332000,8.531709e-01,24.798646,...,0.0000,1.594837e+01,1.594837e+01,9.999999e-01,395.498008,5.932470e+01,0.0,0,0.00,0.0000
3,003554e2dce176b5555353e4f3555ac8,0,0.000000,136.713588,0.000000,1,120.00,120.000000,1.000000e+00,120.000000,...,0.0000,2.794427e-01,2.794427e-01,2.437950e-01,33.533129,5.029969e+00,0.0,0,0.00,0.0000
4,004c9cd9d87a3c30c522c48c4fc07416,153,444.125498,458.559317,124.838170,154,19220.23,124.806688,8.912002e-01,124.885858,...,52.4250,2.629673e+01,2.629673e+01,1.000000e+00,3284.089090,4.926134e+02,2.0,1,349.50,52.4250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2243,ffc470761de7d0232558ba5e786e57b7,10,59.712199,69.217106,75.443000,11,784.33,71.302727,9.475209e-01,76.227925,...,63.6840,1.154270e+01,1.154270e+01,9.999903e-01,879.876244,1.319814e+02,11.0,1,530.22,79.5330
2244,ffdd9f82b9a447f6f8d4b91554cc7dd3,13,413.302558,421.318426,137.730769,14,1890.40,135.028571,9.871178e-01,138.280316,...,1.9500,2.775199e+00,2.775199e+00,9.376629e-01,383.755352,5.756330e+01,3.0,1,111.20,16.6800
2245,ffeee66ac5d5a62fe688b9d26f83f534,12,151.146111,212.335451,131.656667,13,1709.87,131.528462,5.915797e-01,132.257712,...,19.4985,2.996460e+00,2.996460e+00,9.500364e-01,396.304978,5.944575e+01,1.0,1,129.99,19.4985
2246,fffd5413c0700ac820c7069d66d98c89,40,277.587824,280.115324,149.510000,41,6242.90,152.265854,9.950029e-01,149.685363,...,228.1050,1.251972e+01,1.251972e+01,9.999963e-01,1874.019022,2.811029e+02,12.0,1,1806.70,271.0050


In [5]:
df.columns

Index(['seller_id', 'frequency', 'recency', 'T', 'monetary_value',
       'total_orders_cal', 'total_gmv_cal', 'historical_aov_cal', 'prob_alive',
       'expected_avg_gmv', 'expected_orders_30d', 'activity_score_30d',
       'p_purchase_approx_30d', 'predicted_gmv_30d', 'expected_commission_30d',
       'actual_orders_30d', 'actual_active_30d', 'actual_gmv_30d',
       'actual_commission_30d', 'expected_orders_60d', 'activity_score_60d',
       'p_purchase_approx_60d', 'predicted_gmv_60d', 'expected_commission_60d',
       'actual_orders_60d', 'actual_active_60d', 'actual_gmv_60d',
       'actual_commission_60d', 'expected_orders_90d', 'activity_score_90d',
       'p_purchase_approx_90d', 'predicted_gmv_90d', 'expected_commission_90d',
       'actual_orders_90d', 'actual_active_90d', 'actual_gmv_90d',
       'actual_commission_90d'],
      dtype='str')

In [7]:
# 1. Tạo percentile rank cho từng biến
df["rank_expected_commission"] = df["expected_commission_60d"].rank(
    method="average",
    pct=True
)

df["rank_expected_order"] = df["expected_orders_60d"].rank(
    method="average",
    pct=True
)

df["rank_probability_active"] = df["p_purchase_approx_60d"].rank(
    method="average",
    pct=True
)

# 2. Priority Score: chọn seller nên chăm sóc tăng trưởng
df["priority_score"] = (
    0.5 * df["rank_expected_commission"]
    + 0.3 * df["rank_expected_order"]
    + 0.2 * df["rank_probability_active"]
)

# 3. Save Priority / Churn Saving Score:
# seller có commission cao nhưng xác suất inactive cũng cao
df["rank_inactive_risk"] = (1 - df["p_purchase_approx_60d"]).rank(
    method="average",
    pct=True
)

df["save_priority_score"] = (
    df["rank_expected_commission"]
    * df["rank_inactive_risk"]
)

# 4. Gắn cờ top 10% seller nên chăm sóc tăng trưởng
threshold_priority = df["priority_score"].quantile(0.90)
df["top_10_growth_target"] = df["priority_score"] >= threshold_priority

# 5. Gắn cờ top 10% seller nên win-back/chống churn
threshold_save = df["save_priority_score"].quantile(0.90)
df["top_10_save_target"] = df["save_priority_score"] >= threshold_save

# 6. Xem kết quả
df[
    [
        "seller_id",
        "expected_orders_60d",
        "expected_commission_60d",
        "p_purchase_approx_60d",
        "priority_score",
        "save_priority_score",
        "top_10_growth_target",
        "top_10_save_target"
    ]
].sort_values("priority_score", ascending=False).head(20)

,seller_id,expected_orders_60d,expected_commission_60d,p_purchase_approx_60d,priority_score,save_priority_score,top_10_growth_target,top_10_save_target
1126,7d13fca15225358621be4086e1eb0964,164.674718,6040.362710,1.0,0.996486,0.014012,True,False
660,4a3ca9315b744ce9f8e9374361493884,183.559606,3108.288636,1.0,0.996219,0.013994,True,False
642,4869f7a5dfa277a7dca6462dcf3b52b2,114.443830,3734.485897,1.0,0.995596,0.014006,True,False
1123,7c67e1448b00f6e969d365cea6b010ab,107.645511,3101.451486,1.0,0.994795,0.013988,True,False
1921,da8622b14eb17ae2831f4ac5b9dab84a,133.673581,2508.018504,1.0,0.994440,0.013963,True,False
1331,955fee9216a65b617aa5c0531780ce60,175.696574,2419.289998,1.0,0.994395,0.013950,True,False
1099,7a67c85e85bb2ce8582c35f2203ad736,133.163164,2471.969289,1.0,0.994084,0.013956,True,False
280,1f50f920176fa81dab994f9023523100,179.174807,2044.276582,1.0,0.993194,0.013913,True,False
1810,cc419e0650a3c5ba77189a1882b7556a,196.985100,1799.305955,1.0,0.992794,0.013894,True,False
153,1025f0e2d44d7041d6cf58b6550e0bfa,95.507773,2193.973408,1.0,0.992660,0.013931,True,False


In [8]:
df.to_csv(r'outputs\seller_scoring_bgnbd_gammagamma_top_k.csv', index=False)